In [1]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

2026-01-06 16:20:38.333106: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-06 16:20:39.520914: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


cuda


In [2]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH
).to(device)
model.eval()

print(model)

model.load_state_dict(torch.load("best_metric_model.pth"))

UNet(
  (model): Sequential(
    (0): ResidualUnit(
      (conv): Sequential(
        (unit0): Convolution(
          (conv): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
        (unit1): Convolution(
          (conv): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
      )
      (residual): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    )
    (1): SkipConnection(
      (submodule): Sequential(
        (0): ResidualUnit(
          (conv): Sequential(


<All keys matched successfully>

In [3]:
from dataloading.load_atlas import load_atlas

train_id_loader, val_id_loader, test_id_loader = load_atlas()

Found 655 image files
Found 655 mask files
Found 300 test image files
Train data: 524 files
Validation data: 131 files


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [4]:
batch = next(iter(train_id_loader))
image = batch["image"].to(device)
label = batch["label"].to(device)
print(f"Image shape: {image.shape}, Label shape: {label.shape}")


Image shape: torch.Size([1, 1, 197, 233, 189]), Label shape: torch.Size([1, 1, 197, 233, 189])


In [6]:
import torch.nn.functional as F

def get_penultimate_features(model, image):
    with torch.no_grad():
        image = F.interpolate(image, size=(96, 96, 96), mode='trilinear', align_corners=False)
        features = model.model[0](image)
        features = model.model[1](features)
        return features

features = get_penultimate_features(model, image)
features.shape

torch.Size([1, 32, 48, 48, 48])

In [7]:
import torch.nn.functional as F

CLASSES = [0, 1]

class GaussianMeanCalculator:
    def __init__(self):
        self.class_accumulator = {c: torch.zeros(32, dtype=torch.float64).to(device) for c in CLASSES}
        self.class_counts = {c: 0 for c in CLASSES}
    
    def update(self, image, label):
        features = get_penultimate_features(model, image)
        label_interpolated = F.interpolate(label, size=(features.shape[2:5]), mode='nearest').long()

        features_flat = features.permute(0, 2, 3, 4, 1).reshape(-1, 32)
        labels_flat = label_interpolated.reshape(-1)

        for c in CLASSES:
            self.class_accumulator[c] += features_flat[labels_flat == c].sum(dim=0)
            self.class_counts[c] += (labels_flat == c).sum().item()
    
    def compute_means(self):
        means = {c: self.class_accumulator[c] / self.class_counts[c] for c in CLASSES}
        return means

In [8]:
from tqdm import tqdm

def calculate_global_class_means(loader):
    calculator = GaussianMeanCalculator()

    for batch in tqdm(loader):
        image = batch["image"].to(device)
        label = batch["label"].to(device)
        calculator.update(image, label)

    means = calculator.compute_means()
    return means

class_means = calculate_global_class_means(train_id_loader)
print(class_means)

100%|██████████| 524/524 [01:09<00:00,  7.50it/s]

{0: metatensor([ 0.2534,  0.3448,  0.2361, -0.0391,  0.3396,  0.3369,  0.3215,  0.1538,
         0.2461,  0.0892,  0.2018,  0.1046,  0.0359,  0.1766,  0.4020,  0.1486,
         0.7612,  0.7707,  0.7182,  0.8149,  0.7270,  0.6393,  0.8275,  0.7669,
         0.7893,  0.6155,  0.5848,  0.6921,  0.6967,  0.7505,  0.7239,  0.7929],
       device='cuda:0', dtype=torch.float64), 1: metatensor([ 0.1293,  0.4316,  0.2679, -0.0813,  0.2017,  0.3732,  0.2016,  0.1126,
         0.2958, -0.0702,  0.1241,  0.1214,  0.0389,  0.0483,  0.2066,  0.2524,
         0.7394,  0.3597,  1.1786,  0.7220,  0.5374,  1.0417,  3.2448,  1.5677,
         1.5153,  0.7131,  0.3520,  1.1188,  1.5866,  1.6623,  1.4151,  0.6083],
       device='cuda:0', dtype=torch.float64)}


In [9]:
class GaussianTiedCovarianceCalculator:
    def __init__(self, class_means):
        self.class_means = class_means
        self.cov_accumulator = torch.zeros(32, 32, dtype=torch.float64).to(device)
        self.total_count = 0
    
    def update(self, image, label):
        features = get_penultimate_features(model, image)
        label_interpolated = F.interpolate(label, size=(features.shape[2:5]), mode='nearest').long()

        features_flat = features.permute(0, 2, 3, 4, 1).reshape(-1, 32)
        labels_flat = label_interpolated.reshape(-1)

        for c in CLASSES:
            class_features = features_flat[labels_flat == c]
            mean = self.class_means[c].unsqueeze(0)
            diffs = class_features - mean
            self.cov_accumulator += diffs.t() @ diffs
            self.total_count += class_features.size(0)
    
    def compute_covariance(self):
        covariance = self.cov_accumulator / self.total_count
        return covariance

In [10]:
def calculate_global_tied_covariance(loader, class_means):
    calculator = GaussianTiedCovarianceCalculator(class_means)

    for batch in tqdm(loader):
        image = batch["image"].to(device)
        label = batch["label"].to(device)
        calculator.update(image, label)

    covariance = calculator.compute_covariance()
    return covariance

covariance = calculate_global_tied_covariance(train_id_loader, class_means)
print(covariance)

100%|██████████| 524/524 [01:05<00:00,  8.04it/s]

metatensor([[ 0.1020, -0.0828, -0.0678,  ..., -0.0055, -0.0069, -0.0292],
        [-0.0828,  0.4042,  0.1562,  ..., -0.0028,  0.0066,  0.0436],
        [-0.0678,  0.1562,  0.4162,  ...,  0.0019,  0.0081,  0.0287],
        ...,
        [-0.0055, -0.0028,  0.0019,  ...,  1.3503, -0.2251, -0.2490],
        [-0.0069,  0.0066,  0.0081,  ..., -0.2251,  1.4266,  0.5695],
        [-0.0292,  0.0436,  0.0287,  ..., -0.2490,  0.5695,  0.6792]],
       device='cuda:0', dtype=torch.float64)


In [12]:
def calculate_mahalanobis_confidence_score(features, class_means, covariance):
    features_flat = features.permute(0, 2, 3, 4, 1).reshape(-1, 32)
    inv_cov = torch.linalg.pinv(covariance + 1e-6 * torch.eye(covariance.size(0)).to(device))
    scores = []
    for c in CLASSES:
        mean = class_means[c]
        diffs = features_flat - mean
        tmp = diffs @ inv_cov
        score = torch.sum(tmp * diffs, dim=1)
        scores.append(-score)
    scores = torch.stack(scores, dim=0)
    max_scores = torch.max(scores, dim=0).values
    return max_scores.reshape(features.shape[2:])

torch.backends.cuda.preferred_linalg_library('magma')

batch = next(iter(val_id_loader))
image = batch["image"].to(device)
features = get_penultimate_features(model, image)

score = calculate_mahalanobis_confidence_score(features, class_means, covariance)
print(score.shape)


W20260106 16:26:25.378068 140451776773056 Context.cpp:456] Warning: torch.backends.cuda.preferred_linalg_library is an experimental feature. If you see any error or unexpected behavior when this flag is set please file an issue on GitHub. (function operator())


torch.Size([48, 48, 48])


In [17]:
import numpy as np

def calculate_scores(loader):
    scores = []
    for batch in tqdm(loader):
        image = batch["image"].to(device)
        features = get_penultimate_features(model, image)
        score = calculate_mahalanobis_confidence_score(features, class_means, covariance)
        scores.append(score.mean().item())
    return scores

val_scores = calculate_scores(val_id_loader)
threshold = np.percentile(val_scores, 5)
threshold

100%|██████████| 131/131 [00:19<00:00,  6.81it/s]


np.float64(-35.00530873239266)

In [18]:
test_scores = calculate_scores(test_id_loader)
print(test_scores[:10])

100%|██████████| 300/300 [00:26<00:00, 11.15it/s]

[-31.298795065551783, -30.6291802053809, -31.403746380596274, -34.01782262067002, -31.596310075326787, -33.796595479636025, -33.13367778928245, -30.877650797378017, -33.251632270843054, -29.753023956302528]


In [20]:
from dataloading.load_chaos import load_chaos

chaos_loader = load_chaos()

Found 20 Train CT volumes
Found 20 Test CT volumes
Found 0 Train MR volumes
Found 0 Test MR volumes
Total CHAOS volumes: 40
CHAOS dataloader created with 40 samples


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [21]:
chaos_scores = calculate_scores(chaos_loader)
print(chaos_scores[:10])

100%|██████████| 40/40 [00:57<00:00,  1.43s/it]

[-1574.7203239706505, -1525.0313356193917, -1558.71226584442, -1612.2407760778412, -1404.851947335611, -1493.8442858851793, -1615.3457518402156, -1398.9980804405882, -1401.981543363697, -1751.024481469262]


In [22]:
scores = np.concatenate([np.array(chaos_scores), np.array(test_scores)])
labels = np.concatenate([np.ones_like(chaos_scores), np.zeros_like(test_scores)])

In [23]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = -scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 1.0000, fpr90: 0.0000, fpr95: 0.0000, fpr99: 0.0000


In [24]:
from dataloading.load_brats import load_brats
brats_loader = load_brats()

Found 585 Train BRATS volumes
Found 87 Test BRATS volumes


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [26]:
brats_scores = calculate_scores(brats_loader)
print(brats_scores[:10])

  0%|          | 0/672 [00:00<?, ?it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x56332c1faeb0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00101728

ImageSeriesReader (0x56332c1faeb0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000545123

  1%|          | 5/672 [00:04<08:05,  1.37it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x56332c1faeb0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00100171

  4%|▎         | 25/672 [00:10<02:30,  4.30it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x56332c1faeb0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000545999

ImageSeriesReader (0x56332c1faeb0): Non uniform sampling or missing slices detected,  maximum

[-700.5776378866457, -571.536285956154, -692.4989424922688, -785.518508791079, -785.17249565939, -985.8770654204413, -370.0857616137744, -578.7047616338893, -738.8742195046996, -884.3895001487014]


In [27]:
scores = np.concatenate([np.array(brats_scores), np.array(test_scores)])
labels = np.concatenate([np.ones_like(brats_scores), np.zeros_like(test_scores)])

In [28]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = -scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 1.0000, fpr90: 0.0000, fpr95: 0.0000, fpr99: 0.0000
